# GMLC PSAP Misroute Root-Cause Analysis v4

This is a **single self-contained notebook**. Run **Run All** from the first cell to the end.

It uses only these four raw exports under `C:\temp\gmlc_v2`:

- `e911_lte_master_gmlc` — primary call population and routed PSAP
- `e911_lte_master_ims` — exact call-level signaling enrichment
- `raw_ccdr` — exact IMS event enrichment
- `e911_psap_boundaries1` — PSAP geometry

It does **not** use simulator-derived label files or any local `.py` module. Test and simulated calls remain in the population and receive explicit tags.

### End-to-end outputs

The notebook validates files and headers, reconstructs PSAP polygons, streams and labels every GMLC source row, resolves an explicit one-call-per-`UNIQ911_CID` grain (with row fallback for missing IDs), profiles exact IMS/RAW joins, preserves one output row per canonical call, performs automated RCA, and writes versioned CSV/JSON evidence under `outputs_psap_rca_v4`.

## 0. Configuration and dependencies

Required Python packages: `pandas`, `numpy`, `shapely`, and `pyproj`. Change only the paths or chunk sizes if needed.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import json
import math
import re
import sqlite3
import time
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from pyproj import Geod
from shapely.geometry import Point, Polygon, shape
from shapely.ops import nearest_points, unary_union
from shapely.strtree import STRtree
from shapely import wkt

DATA_DIR = Path(r"C:\temp\gmlc_v2")
OUTPUT_DIR = DATA_DIR / "outputs_psap_rca_v4"
DB_PATH = OUTPUT_DIR / "psap_rca_stage_v4.sqlite"

GMLC_CHUNK_ROWS = 50_000
SOURCE_CHUNK_ROWS = 75_000
BOUNDARY_CHUNK_ROWS = 5_000
EXPORT_CHUNK_ROWS = 100_000
RCA_CHUNK_ROWS = 25_000
MAX_GMLC_ROWS = None  # None = all GMLC rows
REBUILD_STAGING = True
REQUIRE_ALL_FOUR_EXPORTS = True
MIN_JOIN_COVERAGE_WARNING = 0.50

RCA_MIN_STRONG_CALLS = 20
RCA_MIN_MISROUTES = 3
NUMERIC_SAMPLE_PER_CHUNK = 500
NUMERIC_SAMPLE_TOTAL = 5_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR   :", DATA_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("Mode       :", "FULL GMLC" if MAX_GMLC_ROWS is None else f"FIRST {MAX_GMLC_ROWS:,} GMLC ROWS")


## 1. Resolve the four exports and validate exact headers

File extensions may be hidden in Windows Explorer; both extensionless and `.csv` names are accepted. Required label and join columns fail early. Optional fields are reported and used only when present.

In [ ]:
def canon(value: str) -> str:
    return str(value).replace("\ufeff", "").strip().upper()


def resolve_export(root: Path, names: Sequence[str]) -> Path | None:
    if not root.exists():
        return None
    files = [p for p in root.iterdir() if p.is_file()]
    by_name = {p.name.lower(): p for p in files}
    for name in names:
        key = name.lower()
        for candidate in (key, key + ".csv" if not key.endswith(".csv") else key[:-4]):
            if candidate in by_name:
                return by_name[candidate]
    stems = {Path(n).stem.lower() for n in names}
    return next((p for p in files if p.stem.lower() in stems), None)


def read_header(path: Path) -> list[str]:
    return [canon(c) for c in pd.read_csv(path, nrows=0, encoding_errors="replace").columns]


def iter_selected_csv(
    path: Path,
    wanted: Sequence[str],
    chunksize: int,
    nrows: int | None = None,
) -> Iterable[pd.DataFrame]:
    wanted_set = {canon(c) for c in wanted}
    seen = 0
    reader = pd.read_csv(
        path,
        dtype="string",
        chunksize=chunksize,
        usecols=lambda c: canon(c) in wanted_set,
        low_memory=False,
        encoding_errors="replace",
    )
    for chunk in reader:
        chunk.columns = [canon(c) for c in chunk.columns]
        if nrows is not None:
            remaining = nrows - seen
            if remaining <= 0:
                break
            chunk = chunk.iloc[:remaining].copy()
        seen += len(chunk)
        yield chunk


EXPORT_NAMES = {
    "GMLC": ("e911_lte_master_gmlc", "e911_lte_master_gmlc.csv"),
    "IMS": ("e911_lte_master_ims", "e911_lte_master_ims.csv"),
    "RAW_CCDR": ("raw_ccdr", "raw_ccdr.csv"),
    "BOUNDARIES": ("e911_psap_boundaries1", "e911_psap_boundaries1.csv"),
}
paths = {source: resolve_export(DATA_DIR, names) for source, names in EXPORT_NAMES.items()}
if REQUIRE_ALL_FOUR_EXPORTS and any(path is None for path in paths.values()):
    missing = [source for source, path in paths.items() if path is None]
    raise FileNotFoundError(f"Missing required raw exports under {DATA_DIR}: {missing}")

headers = {source: set(read_header(path)) for source, path in paths.items() if path is not None}

GMLC_REQUIRED = {
    "UNIQ911_CID", "CALL_BEGIN_TIME_UTC", "FCC_PSAP_ID",
    "LATITUDE", "LONGITUDE", "UNCERT_METERS", "TEST_CALL", "PSAP_SIM_CALL",
}
IMS_REQUIRED = {"UNIQ911_CID"}
BOUNDARY_GEOMETRY_CANDIDATES = {
    "PROPERTIES", "PROPERTIES_CHUNK", "BOUNDARY_JSON_TEXT", "GEOMETRY", "GEOJSON", "WKT"
}
required_errors = []
if not GMLC_REQUIRED.issubset(headers.get("GMLC", set())):
    required_errors.append(f"GMLC missing {sorted(GMLC_REQUIRED - headers.get('GMLC', set()))}")
if not IMS_REQUIRED.issubset(headers.get("IMS", set())):
    required_errors.append(f"IMS missing {sorted(IMS_REQUIRED - headers.get('IMS', set()))}")
if "FCC_PSAP_ID" not in headers.get("BOUNDARIES", set()):
    required_errors.append("BOUNDARIES missing FCC_PSAP_ID")
if not (headers.get("BOUNDARIES", set()) & BOUNDARY_GEOMETRY_CANDIDATES):
    required_errors.append("BOUNDARIES has no recognized geometry/GeoJSON column")
raw_has_icid = "ICID" in headers.get("RAW_CCDR", set())
raw_has_session = "SESSION_ID" in headers.get("RAW_CCDR", set())
ims_has_icid = "IMSCHARGINGID" in headers.get("IMS", set())
ims_has_session = "SESSIONID" in headers.get("IMS", set())
if not ((raw_has_icid and ims_has_icid) or (raw_has_session and ims_has_session)):
    required_errors.append("IMS/RAW share neither IMSCHARGINGID=ICID nor SESSIONID=SESSION_ID")
if required_errors:
    raise ValueError("; ".join(required_errors))

file_status = pd.DataFrame([
    {
        "SOURCE": source,
        "FOUND": path is not None,
        "FILE": str(path or ""),
        "SIZE_GB": round(path.stat().st_size / 1024**3, 3) if path else None,
        "COLUMNS": len(headers.get(source, set())),
    }
    for source, path in paths.items()
])
display(file_status)


In [ ]:
# Exact column names taken from the supplied metadata. Missing optional
# fields are recorded; the loader reads only the intersection.
GMLC_ID_COLS = [
    "UNIQ911_CID", "CTID", "HDR_TRID", "CALLID", "SIP_LINK_ID",
    "SIP_ORIGINAL_CTID", "SIP_CTID", "SETUP_ECGI_HEX", "INVITE_ECGI_HEX",
    "IMSI", "MSISDN", "ESRK", "USID", "FCC_PSAP_ID", "PSAP_ID", "PSAPTN",
]
GMLC_CAT_COLS = [
    "REGION", "MARKET", "MARKET_CLUSTER", "STATE", "COUNTY", "CLMARKET",
    "GMLC_VENDOR", "MME_NAME", "MME_VENDOR", "MME_POOL_ID", "MME_POOL_NAME",
    "SECTOR_NAME", "TAC", "SETUP_ECGI_DEC", "SETUP_ECGI_HEX", "ESRK", "USID",
    "UE_MAKE_ABBR", "UE_MODEL", "PSAP_NAME", "PSAP_NAME_ESRK", "ROUTE_PSAP_NAME",
    "CALL_ANOMALY", "CALL_STRATEGY", "CALL_TRIGGER", "CALL_TYPE", "COMPLETE_CALL",
    "DEFAULT_ROUTED_CALL", "TAC_ROUTED_CALL", "ELS_AVAILABLE", "ELS_USED",
    "EXCEPTION_FLAG", "EXCEPTION_TYPE", "FAILURE_SHORT_DR", "LSR_EXCLUDE_REASON",
    "HDV_RESULT_CODE", "LRRINVITE_STATUS", "MLLSTATUS_CODE", "NW_METHOD_USED",
    "ORIG_RESULT_CODE", "P2_FAILURE_SHORT", "P2_LOCATE", "P2_SUCCESS", "PHASE",
    "PL_RESULT_CODE", "POS_METHOD_USED", "ROUTE_ESINET", "ROUTE_ESZ",
    "ROUTE_FALLBACK", "ROUTE_LOCATION_POS_SOURCE", "ROUTE_OUT_TYPE",
    "ROUTE_STATUS_GMLC", "SHAPE_TYPE", "SIGNALLING_TYPE", "SIPCELL_TYPE",
    "SIP_STATUS", "SIP_SIP_METHOD", "USER_TYPE", "ENV", "INPUT_FILE_NAME",
    "TEST_CALL", "PSAP_SIM_CALL", "Z_GENERATED", "Z_RELAYED2PSAP",
]
GMLC_NUM_COLS = [
    "CALL_BEGIN_TO_LOCATE_SEC", "CALL_DURATION_SEC", "CANCEL_CNT",
    "DISTCELL2PSAPXY_M", "DISTCELL2XY_M", "DISTXY2PSAPXY_M", "ESPOSREQ_COUNT",
    "HDV_CNT", "INVITE_CNT", "INVITE_TO_END_SEC", "LBR_ATTEMPTED", "LBR_PSAP_DIFF",
    "LBR_SUCCESS", "LOCATE_CNT", "LOCATE_TIME_SEC", "LOCATE_TO_CALL_END_SEC",
    "LRR_INVITE_DELTA_SEC", "ORIG_CNT", "PSL_COUNT", "RTT_INFO", "TIME_ON_LTE_SEC",
    "UNCERT_METERS", "VERTICAL_UNCERT", "CONFIDENCE", "CELL_LATITUDE",
    "CELL_LONGITUDE", "LATITUDE", "LONGITUDE", "LOCATION_FIRST_CONFIDENCE",
    "LOCATION_FIRST_HORIZONTAL_UNCERT", "LOCATION_LAST_CONFIDENCE",
    "LOCATION_LAST_HORIZONTAL_UNCERT", "ESPOSREQ_LOC_CONFIDENCE",
    "ESPOSREQ_LOC_HORIZONTAL_UNCERT",
]
GMLC_PRESENCE_COLS = [
    "CONTACT_HDR", "P_EMERGENCY_INFO", "SIP_URI", "SIP_USERAGENT", "SIP_VIA",
    "LOCATION_FIRST_GNSS_POS_DATA", "LOCATION_FIRST_UTRAN_POS_DATA",
    "LOCATION_LAST_GNSS_POS_DATA", "LOCATION_LAST_UTRAN_POS_DATA",
]
GMLC_TIME_COLS = ["CALL_BEGIN_TIME_UTC", "CALL_END_TIME_UTC", "CALL_DATE", "CALL_DATE_UTC"]

IMS_ID_COLS = [
    "UNIQ911_CID", "SESSIONID", "IMSCHARGINGID", "MO_IMSI", "MO_IMEI",
    "UE_IPADDRESS", "ECGI_DEC", "CALLED_TNS", "CALLINGTN", "DIALED_TN",
]
IMS_CAT_COLS = [
    "RAN_REGION", "RAN_MARKET", "RAN_MARKET_CLUSTER", "CLASSIFICATION",
    "ESINET_PROVIDER", "MSC_CLLI", "PGW_CLLI", "NE_SITE_STRING", "LOCATIONXY_YN",
    "MBLT_WRLN", "ORIG_E_CSCF_1", "ORIG_E_CSCF_2", "ORIG_PCSCF_1",
    "ORIG_PCSCF_2", "ORIG_SCC_AS_1", "ORIG_SCC_AS_2", "TERM_E911_SBC_1",
    "TERM_E911_SBC_2", "REGISTER_PCSCF_REASONCODES", "REGISTER_PCSCF_STATUS",
    "RULESET", "SERVICE", "SIGNATURE", "SIP_METHOD", "SOFTDROPCALL", "UE_MAKE",
    "UE_MODEL", "USER_AGENT", "P_EMERGENCY_INFO",
]
IMS_NUM_COLS = [
    "CDRS_LEGS", "COUNT_OF_REGISTER_PCSCF", "DURATION", "ORIG_E_CSCF_CDRS_COUNT",
    "ORIG_P_CSCF_CDRS_COUNT", "ORIG_SCC_AS_CDRS_COUNT",
    "TERM_E911_I_SBC_CDRS_COUNT", "TAC_DEC", "UE_HANDSET_TAC", "UNCETAINITY_METERS",
]
IMS_PRESENCE_COLS = [
    "PIDFLO_TUPLE", "P_CSCF_HEADERINFO", "P_EMERGENCY_INFO", "REGISTERPANI",
    "SDP_MEDIA_EATF", "SDP_MEDIA_ECSCF", "SDP_MEDIA_ISBC", "SDP_MEDIA_PCSCF",
]
IMS_TIME_COLS = ["CALL_DATETIME", "CALL_DATE_UTC", "STARTTIMEHUMAN", "ENDTIMEHUMAN"]

RAW_ID_COLS = ["CALLED_NUMBER", "ICID", "IMSI", "IMEI", "MSISDN", "SESSION_ID", "UE_IP_ADDR"]
RAW_CAT_COLS = [
    "CORRELATION_STATUS", "DEFAULT_ROUTED_CALL", "ECSCF_STATUS", "END_CELLSITE",
    "END_CELLSITE_TZ", "FILENAME", "NODE_ADDRESS", "NON_REGISTER_PCSCF_STATUS",
    "RECORD_TYPE", "REGISTER_PCSCF_STATUS", "START_CELLSITE", "START_CELLSITE_TZ",
]
RAW_NUM_COLS = ["COUNT_OF_ECSCF", "COUNT_OF_NON_REGISTER_PCSCF", "COUNT_OF_REGISTER_PCSCF"]
RAW_PRESENCE_COLS = [
    "EMERGENCY_INFO", "PCSCF_HEADER_INFO", "PIDF_LO", "REQUESTED_PARTY_ADDRESS",
    "SDP_MEDIA_NAMES", "USER_AGENT_INFO",
]
RAW_TIME_COLS = [
    "CALL_START_DATETIME", "CALL_END_DATETIME", "DATETIME_INS_DBTZ",
    "INVITE_RCVD_DATETIME_UTC_P", "INVITE_RECEIVED_DATETIME",
]

requested_by_source = {
    "GMLC": list(dict.fromkeys(GMLC_ID_COLS + GMLC_CAT_COLS + GMLC_NUM_COLS + GMLC_PRESENCE_COLS + GMLC_TIME_COLS)),
    "IMS": list(dict.fromkeys(IMS_ID_COLS + IMS_CAT_COLS + IMS_NUM_COLS + IMS_PRESENCE_COLS + IMS_TIME_COLS)),
    "RAW_CCDR": list(dict.fromkeys(RAW_ID_COLS + RAW_CAT_COLS + RAW_NUM_COLS + RAW_PRESENCE_COLS + RAW_TIME_COLS)),
}

schema_rows = []
for source, requested in requested_by_source.items():
    for column in requested:
        schema_rows.append({
            "SOURCE": source,
            "COLUMN": column,
            "PRESENT": column in headers[source],
            "REQUIRED": column in ({"GMLC": GMLC_REQUIRED, "IMS": IMS_REQUIRED}.get(source, set())),
        })
schema_validation = pd.DataFrame(schema_rows)
schema_validation.to_csv(OUTPUT_DIR / "schema_validation_v4.csv", index=False)
display(
    schema_validation.groupby("SOURCE")
    .agg(REQUESTED=("COLUMN", "size"), PRESENT=("PRESENT", "sum"), REQUIRED=("REQUIRED", "sum"))
    .reset_index()
)
display(Markdown("### Missing optional columns"))
display(schema_validation.loc[~schema_validation["PRESENT"]].head(100))


## 2. Reconstruct and validate PSAP boundaries

The parser accepts full `PROPERTIES` GeoJSON, direct geometry/WKT, or chunked `PROPERTIES_CHUNK` plus `CHUNK_NO`. Duplicate pieces for the same FCC PSAP are unioned before the spatial index is built.

In [ ]:
def norm_id_value(value) -> str:
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip().strip('"').upper()
    return re.sub(r"\.0$", "", text)


def norm_id_series(series: pd.Series) -> pd.Series:
    return (
        series.astype("string").fillna("").str.strip().str.replace('"', "", regex=False)
        .str.replace(r"\.0$", "", regex=True).str.upper()
    )


def parse_geo_payload(payload):
    if payload is None:
        return None
    if isinstance(payload, str):
        text = payload.strip()
        if not text:
            return None
        try:
            return parse_geo_payload(json.loads(text))
        except Exception:
            try:
                return wkt.loads(text)
            except Exception:
                return None
    if isinstance(payload, list):
        geoms = [parse_geo_payload(item) for item in payload]
        geoms = [g for g in geoms if g is not None and not g.is_empty]
        return unary_union(geoms) if geoms else None
    if not isinstance(payload, dict):
        return None

    geometry_type = str(payload.get("type", "")).upper()
    if geometry_type == "FEATURE":
        return parse_geo_payload(payload.get("geometry"))
    if geometry_type == "FEATURECOLLECTION":
        return parse_geo_payload(payload.get("features", []))
    if geometry_type in {
        "POLYGON", "MULTIPOLYGON", "LINESTRING", "MULTILINESTRING",
        "POINT", "MULTIPOINT", "GEOMETRYCOLLECTION",
    }:
        try:
            return shape(payload)
        except Exception:
            return None
    for key in ("geometry", "geom", "geojson", "payload", "properties"):
        if key in payload:
            geom = parse_geo_payload(payload[key])
            if geom is not None:
                return geom
    return None


def repair_geometry(geom):
    if geom is None or geom.is_empty:
        return None
    try:
        if not geom.is_valid:
            geom = geom.buffer(0)
        return None if geom.is_empty or not geom.is_valid else geom
    except Exception:
        return None


def reconstruct_boundaries(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    header = set(read_header(path))
    geometry_col = next((c for c in [
        "PROPERTIES_CHUNK", "PROPERTIES", "BOUNDARY_JSON_TEXT", "GEOJSON", "GEOMETRY", "WKT"
    ] if c in header), None)
    if geometry_col is None:
        raise ValueError("No supported boundary geometry column")

    wanted = ["FCC_PSAP_ID", "NENA_ID", geometry_col]
    chunked = geometry_col == "PROPERTIES_CHUNK" or "CHUNK_NO" in header
    if chunked and "CHUNK_NO" in header:
        wanted.append("CHUNK_NO")

    raw_records = []
    if chunked:
        pieces = defaultdict(list)
        nena_by_fcc = {}
        for chunk in iter_selected_csv(path, wanted, BOUNDARY_CHUNK_ROWS):
            for row in chunk.itertuples(index=False):
                record = row._asdict()
                fcc = norm_id_value(record.get("FCC_PSAP_ID"))
                nena = norm_id_value(record.get("NENA_ID"))
                seq = pd.to_numeric(record.get("CHUNK_NO", 0), errors="coerce")
                seq = int(seq) if pd.notna(seq) else 0
                text = "" if pd.isna(record.get(geometry_col)) else str(record.get(geometry_col))
                pieces[(fcc, nena)].append((seq, text))
                nena_by_fcc[fcc] = nena
        for (fcc, nena), parts in pieces.items():
            text = "".join(value for _, value in sorted(parts, key=lambda item: item[0]))
            raw_records.append((fcc, nena, text))
    else:
        for chunk in iter_selected_csv(path, wanted, BOUNDARY_CHUNK_ROWS):
            for row in chunk.itertuples(index=False):
                record = row._asdict()
                raw_records.append((
                    norm_id_value(record.get("FCC_PSAP_ID")),
                    norm_id_value(record.get("NENA_ID")),
                    record.get(geometry_col),
                ))

    pieces_by_fcc = defaultdict(list)
    nena_by_fcc = {}
    parse_rows = []
    for fcc, nena, payload in raw_records:
        geom = repair_geometry(parse_geo_payload(payload))
        status = "OK" if geom is not None else "PARSE_ERROR"
        if not fcc:
            status = "MISSING_FCC_ID" if geom is not None else "MISSING_FCC_AND_PARSE_ERROR"
        parse_rows.append({"FCC_PSAP_ID": fcc, "NENA_ID": nena, "BOUNDARY_PARSE_STATUS": status})
        if fcc and geom is not None:
            pieces_by_fcc[fcc].append(geom)
            nena_by_fcc[fcc] = nena

    boundary_rows = []
    for fcc, geoms in pieces_by_fcc.items():
        merged = repair_geometry(unary_union(geoms))
        if merged is not None:
            boundary_rows.append({"FCC_PSAP_ID": fcc, "NENA_ID": nena_by_fcc.get(fcc, ""), "GEOMETRY": merged})

    boundaries = pd.DataFrame(boundary_rows)
    parse_status = pd.DataFrame(parse_rows)
    return boundaries, parse_status


boundary_started = time.perf_counter()
boundaries, boundary_parse_status = reconstruct_boundaries(paths["BOUNDARIES"])
if boundaries.empty:
    raise ValueError("No usable FCC PSAP boundaries were reconstructed")

boundary_parse_status.to_csv(OUTPUT_DIR / "boundary_parse_status_v4.csv", index=False)
boundary_index = STRtree(boundaries["GEOMETRY"].tolist())
boundary_geometries = boundaries["GEOMETRY"].tolist()
boundary_fcc = boundaries["FCC_PSAP_ID"].tolist()
boundary_by_fcc = dict(zip(boundary_fcc, boundary_geometries))
geometry_position = {id(geom): i for i, geom in enumerate(boundary_geometries)}
geod = Geod(ellps="WGS84")

print(f"Usable FCC PSAP boundaries: {len(boundaries):,}")
print(f"Boundary reconstruction time: {time.perf_counter() - boundary_started:,.1f} sec")
display(
    boundary_parse_status["BOUNDARY_PARSE_STATUS"].value_counts(dropna=False)
    .rename_axis("STATUS").reset_index(name="ROWS")
)


## 3. Conservative call labels from raw GMLC

Each raw GMLC row receives a permanent `GMLC_ROW_ID`; this is the row-preservation key. The location uncertainty area is compared with the routed boundary and alternate PSAP boundaries.

- `DEFINITE_CORRECT`: the uncertainty area is contained by the routed PSAP only.
- `DEFINITE_MISROUTE`: it does not intersect the routed PSAP and is contained by exactly one alternate PSAP.
- Other statuses remain explicit and are not forced into the binary label.

In [ ]:
def candidate_positions(geometry) -> list[int]:
    hits = boundary_index.query(geometry)
    positions = []
    for hit in hits:
        if isinstance(hit, (int, np.integer)):
            positions.append(int(hit))
        else:
            position = geometry_position.get(id(hit))
            if position is not None:
                positions.append(position)
    return positions


def geodesic_uncertainty_polygon(lon: float, lat: float, meters: float, vertices: int = 24):
    if not np.isfinite(meters) or meters <= 0:
        return None
    azimuths = np.linspace(0.0, 360.0, vertices, endpoint=False)
    lons, lats, _ = geod.fwd(
        np.full(vertices, lon), np.full(vertices, lat), azimuths, np.full(vertices, meters)
    )
    polygon = Polygon(zip(lons, lats))
    return repair_geometry(polygon)


def distance_to_boundary_m(lon: float, lat: float, geometry) -> float | None:
    try:
        point = Point(lon, lat)
        _, nearest = nearest_points(point, geometry.boundary)
        _, _, distance = geod.inv(lon, lat, nearest.x, nearest.y)
        return float(abs(distance))
    except Exception:
        return None


def label_one_call(lat_value, lon_value, uncert_value, routed_value) -> tuple:
    lat = pd.to_numeric(lat_value, errors="coerce")
    lon = pd.to_numeric(lon_value, errors="coerce")
    uncert = pd.to_numeric(uncert_value, errors="coerce")
    routed = norm_id_value(routed_value)

    if pd.isna(lat) or pd.isna(lon):
        return "MISSING_LOCATION", None, None, None, None
    lat, lon = float(lat), float(lon)
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return "INVALID_LOCATION", None, None, None, None
    if pd.isna(uncert) or float(uncert) <= 0:
        return "INVALID_UNCERTAINTY", None, None, None, None
    routed_geom = boundary_by_fcc.get(routed)
    if routed_geom is None:
        return "MISSING_ROUTED_BOUNDARY", None, False, None, None

    uncertainty = geodesic_uncertainty_polygon(lon, lat, float(uncert))
    if uncertainty is None:
        return "INVALID_UNCERTAINTY", None, None, None, None

    routed_intersects = bool(routed_geom.intersects(uncertainty))
    routed_contains = bool(routed_geom.covers(uncertainty))
    contains = []
    overlaps = []
    for position in candidate_positions(uncertainty):
        geom = boundary_geometries[position]
        fcc = boundary_fcc[position]
        if geom.covers(uncertainty):
            contains.append(fcc)
        if geom.intersects(uncertainty):
            overlaps.append(fcc)
    contains = sorted(set(contains))
    overlaps = sorted(set(overlaps))
    alternate_contains = [fcc for fcc in contains if fcc != routed]
    alternate_overlaps = [fcc for fcc in overlaps if fcc != routed]

    overlap_set = set(overlaps)
    contain_set = set(contains)

    # Strong labels require a unique spatial answer over the *entire*
    # uncertainty polygon. Any alternate overlap makes the row
    # indeterminate, even when the alternate does not fully contain it.
    if routed_contains and overlap_set == {routed}:
        status, expected = "DEFINITE_CORRECT", routed
    elif (
        (not routed_intersects)
        and len(alternate_contains) == 1
        and overlap_set == {alternate_contains[0]}
        and contain_set == {alternate_contains[0]}
    ):
        status, expected = "DEFINITE_MISROUTE", alternate_contains[0]
    elif len(overlaps) > 1 or (routed_intersects and alternate_overlaps):
        status = "BOUNDARY_AMBIGUOUS"
        expected = alternate_contains[0] if len(alternate_contains) == 1 else None
    else:
        point = Point(lon, lat)
        point_alternates = [
            boundary_fcc[pos] for pos in candidate_positions(point)
            if boundary_fcc[pos] != routed and boundary_geometries[pos].covers(point)
        ]
        if len(set(point_alternates)) == 1:
            status, expected = "POINT_MISMATCH_LOW_CONFIDENCE", point_alternates[0]
        elif not overlaps:
            status, expected = "NO_CONTAINING_PSAP", None
        else:
            status = "BOUNDARY_AMBIGUOUS"
            expected = alternate_contains[0] if len(alternate_contains) == 1 else None

    distance = distance_to_boundary_m(lon, lat, routed_geom)
    label = 1 if status == "DEFINITE_MISROUTE" else (0 if status == "DEFINITE_CORRECT" else None)
    return status, expected, routed_intersects, distance, label


def truthy(series: pd.Series) -> pd.Series:
    return norm_id_series(series).isin({"1", "Y", "YES", "TRUE", "T", "TEST", "SIM", "SIMULATED", "PSAP SIM"})


def tag_test_sim(chunk: pd.DataFrame) -> pd.DataFrame:
    out = chunk.copy()
    is_test = truthy(out["TEST_CALL"]) if "TEST_CALL" in out else pd.Series(False, index=out.index)
    is_sim = truthy(out["PSAP_SIM_CALL"]) if "PSAP_SIM_CALL" in out else pd.Series(False, index=out.index)
    controlled = ["CALL_TYPE", "USER_TYPE", "ENV", "INPUT_FILE_NAME", "PSAP_NAME"]
    possible = pd.Series(False, index=out.index)
    text_reason = pd.Series("", index=out.index, dtype="string")
    pattern = re.compile(r"(?:^|[^A-Z])(?:TEST|SIMULATED|PSAP[ _-]*SIM)(?:[^A-Z]|$)", re.I)
    for column in controlled:
        if column not in out:
            continue
        hit = out[column].astype("string").fillna("").str.contains(pattern, regex=True)
        possible |= hit
        text_reason = text_reason.mask(hit & text_reason.eq(""), column)
    out["IS_TEST_CALL"] = is_test.astype("Int8")
    out["IS_PSAP_SIM_CALL"] = is_sim.astype("Int8")
    out["IS_TEST_OR_SIM"] = (is_test | is_sim | possible).astype("Int8")
    out["TEST_SIM_REASON"] = np.select(
        [is_test & is_sim, is_test, is_sim, possible],
        ["TEST_CALL+PSAP_SIM_CALL", "TEST_CALL", "PSAP_SIM_CALL", "CONTROLLED_TEXT:" + text_reason],
        default="NONE",
    )
    out["CALL_POPULATION_TAG"] = np.select(
        [is_test & is_sim, is_test, is_sim, possible],
        ["TEST_AND_SIM", "TEST", "PSAP_SIM", "POSSIBLE_TEST_SIM"],
        default="STANDARD_OR_UNSPECIFIED",
    )
    return out


In [ ]:
if REBUILD_STAGING and DB_PATH.exists():
    DB_PATH.unlink()
con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA journal_mode=WAL")
con.execute("PRAGMA synchronous=NORMAL")
con.execute("PRAGMA temp_store=FILE")

gmlc_wanted = requested_by_source["GMLC"]
input_gmlc_rows = 0
staged_gmlc_rows = 0
started = time.perf_counter()

for chunk_number, chunk in enumerate(iter_selected_csv(
    paths["GMLC"], gmlc_wanted, GMLC_CHUNK_ROWS, nrows=MAX_GMLC_ROWS
), start=1):
    input_gmlc_rows += len(chunk)
    chunk = tag_test_sim(chunk)
    chunk["GMLC_ROW_ID"] = np.arange(staged_gmlc_rows + 1, staged_gmlc_rows + len(chunk) + 1, dtype=np.int64)
    chunk["GMLC_JOIN_KEY"] = norm_id_series(chunk["UNIQ911_CID"])
    chunk["CALL_KEY"] = np.where(
        chunk["GMLC_JOIN_KEY"].ne(""),
        "CID:" + chunk["GMLC_JOIN_KEY"],
        "ROW:" + chunk["GMLC_ROW_ID"].astype("string"),
    )
    chunk["CALL_GRAIN_METHOD"] = np.where(
        chunk["GMLC_JOIN_KEY"].ne(""), "UNIQ911_CID", "ROW_FALLBACK_MISSING_CID"
    )
    chunk["FCC_PSAP_ID"] = norm_id_series(chunk["FCC_PSAP_ID"])

    labels = [
        label_one_call(lat, lon, unc, routed)
        for lat, lon, unc, routed in zip(
            chunk["LATITUDE"], chunk["LONGITUDE"], chunk["UNCERT_METERS"], chunk["FCC_PSAP_ID"]
        )
    ]
    label_frame = pd.DataFrame(labels, columns=[
        "ROUTE_INTEGRITY_STATUS", "EXPECTED_FCC_PSAP_ID",
        "ROUTED_INTERSECTS_UNCERT_AREA", "DISTANCE_TO_ROUTED_BOUNDARY_M", "MISROUTE_LABEL",
    ], index=chunk.index)
    chunk = pd.concat([chunk, label_frame], axis=1)
    chunk["MISROUTE_LABEL"] = pd.array(chunk["MISROUTE_LABEL"], dtype="Int8")
    # pandas' default executemany path avoids SQLite's wide-table
    # parameter limit, which method="multi" can exceed on GMLC.
    chunk.to_sql("gmlc_rows_v4", con, if_exists="append", index=False, chunksize=5_000)
    staged_gmlc_rows += len(chunk)

    if chunk_number == 1 or staged_gmlc_rows % 250_000 < len(chunk):
        elapsed = time.perf_counter() - started
        print(f"Labeled {staged_gmlc_rows:,} GMLC rows in {elapsed:,.1f} sec")

con.execute("CREATE UNIQUE INDEX ux_gmlc_source_row_v4 ON gmlc_rows_v4(GMLC_ROW_ID)")
con.execute("CREATE INDEX ix_gmlc_source_call_v4 ON gmlc_rows_v4(CALL_KEY)")
con.execute("CREATE INDEX ix_gmlc_source_cid_v4 ON gmlc_rows_v4(GMLC_JOIN_KEY)")
con.commit()

staged_check = con.execute("SELECT COUNT(*) FROM gmlc_rows_v4").fetchone()[0]
if input_gmlc_rows != staged_gmlc_rows or staged_gmlc_rows != staged_check:
    raise AssertionError("GMLC row preservation failed during label staging")

# Canonical call grain: one row per normalized UNIQ911_CID. Rows with a
# missing CID remain separate through a permanent ROW:<GMLC_ROW_ID>
# fallback. Duplicate source rows are audited before selecting the first
# representative row. Any disagreement in label-defining fields removes
# the call from the strong-label population.
con.execute("DROP TABLE IF EXISTS call_grain_audit_v4")
con.execute(
    """
    CREATE TABLE call_grain_audit_v4 AS
    SELECT
        CALL_KEY,
        MIN(GMLC_ROW_ID) AS REPRESENTATIVE_GMLC_ROW_ID,
        MIN(GMLC_JOIN_KEY) AS GMLC_JOIN_KEY,
        MIN(CALL_GRAIN_METHOD) AS CALL_GRAIN_METHOD,
        COUNT(*) AS SOURCE_ROW_COUNT,
        COUNT(DISTINCT COALESCE(UNIQ911_CID, '<NULL>')) AS RAW_CID_VARIANTS,
        COUNT(DISTINCT COALESCE(FCC_PSAP_ID, '<NULL>')) AS ROUTED_PSAP_VARIANTS,
        COUNT(DISTINCT COALESCE(LATITUDE, '<NULL>')) AS LATITUDE_VARIANTS,
        COUNT(DISTINCT COALESCE(LONGITUDE, '<NULL>')) AS LONGITUDE_VARIANTS,
        COUNT(DISTINCT COALESCE(UNCERT_METERS, '<NULL>')) AS UNCERTAINTY_VARIANTS,
        COUNT(DISTINCT COALESCE(ROUTE_INTEGRITY_STATUS, '<NULL>')) AS LABEL_STATUS_VARIANTS,
        COUNT(DISTINCT COALESCE(EXPECTED_FCC_PSAP_ID, '<NULL>')) AS EXPECTED_PSAP_VARIANTS,
        MAX(COALESCE(IS_TEST_CALL, 0)) AS ANY_TEST_CALL,
        MAX(COALESCE(IS_PSAP_SIM_CALL, 0)) AS ANY_PSAP_SIM_CALL,
        MAX(COALESCE(IS_TEST_OR_SIM, 0)) AS ANY_TEST_OR_SIM
    FROM gmlc_rows_v4
    GROUP BY CALL_KEY
    """
)
con.execute("ALTER TABLE call_grain_audit_v4 ADD COLUMN CALL_CONFLICT INTEGER DEFAULT 0")
con.execute("ALTER TABLE call_grain_audit_v4 ADD COLUMN CONFLICT_REASON TEXT")
con.execute(
    """
    UPDATE call_grain_audit_v4
    SET CALL_CONFLICT = CASE WHEN
            RAW_CID_VARIANTS > 1 OR ROUTED_PSAP_VARIANTS > 1 OR LATITUDE_VARIANTS > 1 OR LONGITUDE_VARIANTS > 1
            OR UNCERTAINTY_VARIANTS > 1 OR LABEL_STATUS_VARIANTS > 1
            OR EXPECTED_PSAP_VARIANTS > 1
        THEN 1 ELSE 0 END,
        CONFLICT_REASON = CASE WHEN
            RAW_CID_VARIANTS > 1 OR ROUTED_PSAP_VARIANTS > 1 OR LATITUDE_VARIANTS > 1 OR LONGITUDE_VARIANTS > 1
            OR UNCERTAINTY_VARIANTS > 1 OR LABEL_STATUS_VARIANTS > 1
            OR EXPECTED_PSAP_VARIANTS > 1
        THEN 'DUPLICATE_CID_DISAGREES_ON_LABEL_INPUT_OR_RESULT' ELSE '' END
    """
)
con.execute("CREATE UNIQUE INDEX ux_call_grain_key_v4 ON call_grain_audit_v4(CALL_KEY)")

con.execute("DROP TABLE IF EXISTS gmlc_calls_v4")
con.execute(
    """
    CREATE TABLE gmlc_calls_v4 AS
    SELECT r.*
    FROM gmlc_rows_v4 r
    JOIN call_grain_audit_v4 a
      ON a.REPRESENTATIVE_GMLC_ROW_ID = r.GMLC_ROW_ID
    """
)
for definition in (
    "SOURCE_ROW_COUNT INTEGER", "CALL_CONFLICT INTEGER", "CONFLICT_REASON TEXT"
):
    con.execute(f"ALTER TABLE gmlc_calls_v4 ADD COLUMN {definition}")
con.execute(
    """
    UPDATE gmlc_calls_v4
    SET SOURCE_ROW_COUNT = (SELECT a.SOURCE_ROW_COUNT FROM call_grain_audit_v4 a WHERE a.CALL_KEY=gmlc_calls_v4.CALL_KEY),
        CALL_CONFLICT = (SELECT a.CALL_CONFLICT FROM call_grain_audit_v4 a WHERE a.CALL_KEY=gmlc_calls_v4.CALL_KEY),
        CONFLICT_REASON = (SELECT a.CONFLICT_REASON FROM call_grain_audit_v4 a WHERE a.CALL_KEY=gmlc_calls_v4.CALL_KEY),
        IS_TEST_CALL = (SELECT a.ANY_TEST_CALL FROM call_grain_audit_v4 a WHERE a.CALL_KEY=gmlc_calls_v4.CALL_KEY),
        IS_PSAP_SIM_CALL = (SELECT a.ANY_PSAP_SIM_CALL FROM call_grain_audit_v4 a WHERE a.CALL_KEY=gmlc_calls_v4.CALL_KEY),
        IS_TEST_OR_SIM = (SELECT a.ANY_TEST_OR_SIM FROM call_grain_audit_v4 a WHERE a.CALL_KEY=gmlc_calls_v4.CALL_KEY)
    """
)
con.execute(
    """
    UPDATE gmlc_calls_v4
    SET CALL_POPULATION_TAG = CASE
            WHEN IS_TEST_CALL=1 AND IS_PSAP_SIM_CALL=1 THEN 'TEST_AND_SIM'
            WHEN IS_TEST_CALL=1 THEN 'TEST'
            WHEN IS_PSAP_SIM_CALL=1 THEN 'PSAP_SIM'
            WHEN IS_TEST_OR_SIM=1 THEN 'POSSIBLE_TEST_SIM'
            ELSE 'STANDARD_OR_UNSPECIFIED' END,
        TEST_SIM_REASON = CASE
            WHEN IS_TEST_CALL=1 AND IS_PSAP_SIM_CALL=1 THEN 'TEST_CALL+PSAP_SIM_CALL'
            WHEN IS_TEST_CALL=1 THEN 'TEST_CALL'
            WHEN IS_PSAP_SIM_CALL=1 THEN 'PSAP_SIM_CALL'
            WHEN IS_TEST_OR_SIM=1 THEN 'CONTROLLED_TEXT_ON_ONE_OR_MORE_SOURCE_ROWS'
            ELSE 'NONE' END
    """
)
con.execute(
    """
    UPDATE gmlc_calls_v4
    SET ROUTE_INTEGRITY_STATUS='CONFLICTING_GMLC_ROWS',
        EXPECTED_FCC_PSAP_ID=NULL,
        ROUTED_INTERSECTS_UNCERT_AREA=NULL,
        DISTANCE_TO_ROUTED_BOUNDARY_M=NULL,
        MISROUTE_LABEL=NULL
    WHERE CALL_CONFLICT=1
    """
)
con.execute("CREATE UNIQUE INDEX ux_gmlc_call_v4 ON gmlc_calls_v4(CALL_KEY)")
con.execute("CREATE INDEX ix_gmlc_call_cid_v4 ON gmlc_calls_v4(GMLC_JOIN_KEY)")
con.commit()

staged_call_count = con.execute("SELECT COUNT(*) FROM gmlc_calls_v4").fetchone()[0]
expected_call_count = con.execute(
    """SELECT COUNT(DISTINCT CASE WHEN GMLC_JOIN_KEY<>'' THEN GMLC_JOIN_KEY END)
              + SUM(CASE WHEN GMLC_JOIN_KEY='' THEN 1 ELSE 0 END)
       FROM gmlc_rows_v4"""
).fetchone()[0]
duplicate_call_keys = con.execute(
    "SELECT COUNT(*) FROM (SELECT CALL_KEY FROM gmlc_calls_v4 GROUP BY CALL_KEY HAVING COUNT(*)>1)"
).fetchone()[0]
conflict_strong_labels = con.execute(
    "SELECT COUNT(*) FROM gmlc_calls_v4 WHERE CALL_CONFLICT=1 AND MISROUTE_LABEL IS NOT NULL"
).fetchone()[0]
if staged_call_count != expected_call_count or duplicate_call_keys or conflict_strong_labels:
    raise AssertionError("Canonical call-grain QA failed")

call_grain_path = OUTPUT_DIR / "call_grain_audit_v4.csv"
if call_grain_path.exists():
    call_grain_path.unlink()
first_audit_chunk = True
for audit_chunk in pd.read_sql_query(
    "SELECT * FROM call_grain_audit_v4 ORDER BY SOURCE_ROW_COUNT DESC, CALL_KEY",
    con,
    chunksize=EXPORT_CHUNK_ROWS,
):
    audit_chunk.to_csv(
        call_grain_path,
        mode="w" if first_audit_chunk else "a",
        header=first_audit_chunk,
        index=False,
    )
    first_audit_chunk = False
call_grain_preview = pd.read_sql_query(
    "SELECT * FROM call_grain_audit_v4 ORDER BY SOURCE_ROW_COUNT DESC, CALL_KEY LIMIT 25",
    con,
)

label_summary = pd.read_sql_query(
    """SELECT ROUTE_INTEGRITY_STATUS, COUNT(*) AS ROWS,
              ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM gmlc_calls_v4), 4) AS RATE_PCT
       FROM gmlc_calls_v4 GROUP BY ROUTE_INTEGRITY_STATUS ORDER BY ROWS DESC""",
    con,
)
display(label_summary)
display(call_grain_preview)
print(f"Input GMLC rows: {staged_gmlc_rows:,}; canonical calls: {staged_call_count:,}")


## 4. Exact IMS enrichment

The only final GMLC→IMS join is exact `UNIQ911_CID = UNIQ911_CID`. IMS event rows are aggregated to one record per key before attachment, so duplicate source rows cannot multiply GMLC calls.

In [ ]:
def aggregate_source_chunk(
    chunk: pd.DataFrame,
    key: pd.Series,
    source: str,
    id_cols: Sequence[str],
    cat_cols: Sequence[str],
    num_cols: Sequence[str],
    presence_cols: Sequence[str],
    connection,
    categorical_partial_table: str,
) -> pd.DataFrame:
    """Create bounded per-chunk facts; global reduction stays in SQLite.

    Categorical values are stored as per-(call, feature, value) counts so
    the final value is the true global mode, not a mode-of-chunk-modes.
    Numeric facts retain additive count/sum plus global min/max.  No
    chunk median is presented as a call-level median.
    """
    work = chunk.copy()
    work["__KEY"] = key.values
    work = work[work["__KEY"].astype("string").ne("")]
    if work.empty:
        return pd.DataFrame()
    grouped = work.groupby("__KEY", sort=False, dropna=False)
    result = grouped.size().rename(f"{source}__ROW_COUNT").to_frame()

    for column in dict.fromkeys([*id_cols, *cat_cols]):
        if column not in work:
            continue
        values = work[column].astype("string").fillna("").str.strip()
        valid = values.ne("")
        if not valid.any():
            continue
        counts = (
            pd.DataFrame({"__KEY": work.loc[valid, "__KEY"], "FEATURE_VALUE": values.loc[valid]})
            .groupby(["__KEY", "FEATURE_VALUE"], sort=False, dropna=False)
            .size()
            .rename("VALUE_COUNT")
            .reset_index()
        )
        counts.insert(1, "FEATURE", column)
        counts.to_sql(
            categorical_partial_table,
            connection,
            if_exists="append",
            index=False,
            chunksize=5_000,
        )

    for column in dict.fromkeys(num_cols):
        if column not in work:
            continue
        numeric = pd.to_numeric(work[column], errors="coerce")
        temp = pd.DataFrame({"__KEY": work["__KEY"], "VALUE": numeric})
        numeric_group = temp.groupby("__KEY", sort=False)["VALUE"]
        result[f"{source}__{column}__NUM_COUNT_PART"] = numeric_group.count()
        result[f"{source}__{column}__SUM_PART"] = numeric_group.sum(min_count=1)
        result[f"{source}__{column}__MIN_PART"] = numeric_group.min()
        result[f"{source}__{column}__MAX_PART"] = numeric_group.max()
    for column in dict.fromkeys(presence_cols):
        if column not in work:
            continue
        present = work[column].astype("string").fillna("").str.strip().ne("")
        temp = pd.DataFrame({"__KEY": work["__KEY"], "VALUE": present.astype(int)})
        result[f"{source}__{column}__PRESENT"] = temp.groupby("__KEY")["VALUE"].max()
    return result.reset_index()


def quote_ident(name: str) -> str:
    return '"' + str(name).replace('"', '""') + '"'


def table_columns(connection, table: str) -> list[str]:
    return [row[1] for row in connection.execute(f"PRAGMA table_info({quote_ident(table)})").fetchall()]


def create_categorical_partial_table(connection, table: str):
    connection.execute(f"DROP TABLE IF EXISTS {quote_ident(table)}")
    connection.execute(
        f"""CREATE TABLE {quote_ident(table)} (
                __KEY TEXT NOT NULL,
                FEATURE TEXT NOT NULL,
                FEATURE_VALUE TEXT NOT NULL,
                VALUE_COUNT INTEGER NOT NULL
            )"""
    )


def finalize_partial_table(
    connection,
    partial_table: str,
    categorical_partial_table: str,
    output_table: str,
    source: str,
):
    columns = table_columns(connection, partial_table)
    expressions = [quote_ident("__KEY")]
    if f"{source}__ROW_COUNT" in columns:
        expressions.append(
            f"SUM({quote_ident(source + '__ROW_COUNT')}) AS {quote_ident(source + '__ROW_COUNT')}"
        )

    numeric_bases = sorted({
        column[:-len("__NUM_COUNT_PART")]
        for column in columns if column.endswith("__NUM_COUNT_PART")
    })
    for base in numeric_bases:
        count_col = base + "__NUM_COUNT_PART"
        sum_col = base + "__SUM_PART"
        min_col = base + "__MIN_PART"
        max_col = base + "__MAX_PART"
        expressions.extend([
            f"SUM({quote_ident(count_col)}) AS {quote_ident(base + '__COUNT')}",
            f"SUM({quote_ident(sum_col)}) AS {quote_ident(base + '__SUM')}",
            f"1.0 * SUM({quote_ident(sum_col)}) / NULLIF(SUM({quote_ident(count_col)}), 0) "
            f"AS {quote_ident(base + '__MEAN')}",
            f"MIN({quote_ident(min_col)}) AS {quote_ident(base + '__MIN')}",
            f"MAX({quote_ident(max_col)}) AS {quote_ident(base + '__MAX')}",
        ])

    for column in columns:
        if column.endswith("__PRESENT"):
            expressions.append(f"MAX({quote_ident(column)}) AS {quote_ident(column)}")

    connection.execute(f"DROP TABLE IF EXISTS {quote_ident(output_table)}")
    connection.execute(
        f"CREATE TABLE {quote_ident(output_table)} AS SELECT "
        + ", ".join(expressions)
        + f" FROM {quote_ident(partial_table)} GROUP BY {quote_ident('__KEY')}"
    )

    # One disk-backed global mode per call and categorical feature.
    categorical_features = [
        row[0] for row in connection.execute(
            f"SELECT DISTINCT FEATURE FROM {quote_ident(categorical_partial_table)} ORDER BY FEATURE"
        ).fetchall()
    ]
    if categorical_features:
        mode_table = output_table + "__categorical_modes"
        expanded_table = output_table + "__with_categories"
        connection.execute(f"DROP TABLE IF EXISTS {quote_ident(mode_table)}")
        connection.execute(
            f"""CREATE TABLE {quote_ident(mode_table)} AS
                WITH totals AS (
                    SELECT __KEY, FEATURE, FEATURE_VALUE, SUM(VALUE_COUNT) AS TOTAL_VALUE_COUNT
                    FROM {quote_ident(categorical_partial_table)}
                    GROUP BY __KEY, FEATURE, FEATURE_VALUE
                ), ranked AS (
                    SELECT *, ROW_NUMBER() OVER (
                        PARTITION BY __KEY, FEATURE
                        ORDER BY TOTAL_VALUE_COUNT DESC, FEATURE_VALUE ASC
                    ) AS RN
                    FROM totals
                )
                SELECT __KEY, FEATURE, FEATURE_VALUE, TOTAL_VALUE_COUNT
                FROM ranked WHERE RN=1"""
        )
        connection.execute(
            f"CREATE INDEX {quote_ident('ix_' + mode_table + '_key')} "
            f"ON {quote_ident(mode_table)}(__KEY)"
        )
        pivots = ", ".join(
            "MAX(CASE WHEN m.FEATURE="
            + repr(feature)
            + f" THEN m.FEATURE_VALUE END) AS {quote_ident(source + '__' + feature)}"
            for feature in categorical_features
        )
        connection.execute(f"DROP TABLE IF EXISTS {quote_ident(expanded_table)}")
        connection.execute(
            f"""CREATE TABLE {quote_ident(expanded_table)} AS
                SELECT o.*, {pivots}
                FROM {quote_ident(output_table)} o
                LEFT JOIN {quote_ident(mode_table)} m ON m.__KEY=o.__KEY
                GROUP BY o.__KEY"""
        )
        connection.execute(f"DROP TABLE {quote_ident(output_table)}")
        connection.execute(
            f"ALTER TABLE {quote_ident(expanded_table)} RENAME TO {quote_ident(output_table)}"
        )
        connection.execute(f"DROP TABLE {quote_ident(mode_table)}")

    connection.execute(
        f"CREATE UNIQUE INDEX {quote_ident('ux_' + output_table)} "
        f"ON {quote_ident(output_table)}({quote_ident('__KEY')})"
    )
    connection.commit()


def sqlite_membership_mask(
    connection,
    keys: pd.Series,
    lookup_table: str,
    lookup_column: str,
    temp_table: str,
) -> pd.Series:
    """Return a chunk-sized mask; the complete lookup domain stays in SQLite."""
    key_frame = pd.DataFrame({"__KEY": keys.astype("string").drop_duplicates()})
    key_frame = key_frame[key_frame["__KEY"].ne("")]
    key_frame.to_sql(temp_table, connection, if_exists="replace", index=False)
    connection.execute(
        f"CREATE INDEX IF NOT EXISTS {quote_ident('ix_' + temp_table)} "
        f"ON {quote_ident(temp_table)}({quote_ident('__KEY')})"
    )
    matched = pd.read_sql_query(
        f"SELECT DISTINCT k.__KEY FROM {quote_ident(temp_table)} k "
        f"JOIN {quote_ident(lookup_table)} t "
        f"ON t.{quote_ident(lookup_column)}=k.__KEY",
        connection,
    )
    return keys.isin(matched["__KEY"])


eligible_cid_count = con.execute(
    "SELECT COUNT(*) FROM gmlc_calls_v4 WHERE GMLC_JOIN_KEY<>''"
).fetchone()[0]
print(f"Eligible canonical GMLC calls with UNIQ911_CID: {eligible_cid_count:,}")

con.execute("DROP TABLE IF EXISTS ims_partial_v4")
con.execute("DROP TABLE IF EXISTS ims_bridge_pairs_partial_v4")
create_categorical_partial_table(con, "ims_categorical_counts_partial_v4")
ims_wanted = requested_by_source["IMS"]
ims_rows_read = 0
ims_rows_matched = 0
for chunk_number, chunk in enumerate(iter_selected_csv(
    paths["IMS"], ims_wanted, SOURCE_CHUNK_ROWS
), start=1):
    ims_rows_read += len(chunk)
    for column in IMS_ID_COLS:
        if column in chunk:
            chunk[column] = norm_id_series(chunk[column])
    key = norm_id_series(chunk["UNIQ911_CID"])
    mask = sqlite_membership_mask(
        con, key, "gmlc_calls_v4", "GMLC_JOIN_KEY", "ims_chunk_keys_v4"
    )
    ims_rows_matched += int(mask.sum())
    partial = aggregate_source_chunk(
        chunk.loc[mask].copy(), key.loc[mask], "IMS",
        IMS_ID_COLS, IMS_CAT_COLS, IMS_NUM_COLS, IMS_PRESENCE_COLS,
        con, "ims_categorical_counts_partial_v4",
    )
    if not partial.empty:
        partial.to_sql("ims_partial_v4", con, if_exists="append", index=False, chunksize=5_000)

    # Preserve every exact call↔bridge identifier pair. These are not
    # collapsed to a mode/first value before RAW matching.
    pair_frames = []
    for ims_column, raw_key_type in (
        ("IMSCHARGINGID", "ICID"), ("SESSIONID", "SESSION_ID")
    ):
        if ims_column not in chunk:
            continue
        pair = pd.DataFrame({
            "GMLC_JOIN_KEY": key.loc[mask].astype("string"),
            "KEY_TYPE": raw_key_type,
            "RAW_JOIN_KEY": norm_id_series(chunk.loc[mask, ims_column]),
        })
        pair = pair[pair["RAW_JOIN_KEY"].ne("")].drop_duplicates()
        if not pair.empty:
            pair_frames.append(pair)
    if pair_frames:
        pd.concat(pair_frames, ignore_index=True).to_sql(
            "ims_bridge_pairs_partial_v4", con, if_exists="append", index=False, chunksize=5_000
        )
    if chunk_number == 1 or ims_rows_read % 500_000 < len(chunk):
        print(f"IMS read {ims_rows_read:,}; exact-key event matches {ims_rows_matched:,}")

if "ims_partial_v4" not in {row[0] for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}:
    raise ValueError("No IMS rows matched GMLC UNIQ911_CID; exact enrichment cannot continue")
finalize_partial_table(
    con,
    "ims_partial_v4",
    "ims_categorical_counts_partial_v4",
    "ims_agg_v4",
    "IMS",
)

table_names = {row[0] for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
if "ims_bridge_pairs_partial_v4" not in table_names:
    raise ValueError("Matched IMS rows supplied no usable IMSCHARGINGID or SESSIONID bridge keys")
con.execute("DROP TABLE IF EXISTS ims_bridge_pairs_v4")
con.execute(
    """
    CREATE TABLE ims_bridge_pairs_v4 AS
    SELECT DISTINCT c.CALL_KEY, p.KEY_TYPE, p.RAW_JOIN_KEY
    FROM ims_bridge_pairs_partial_v4 p
    JOIN gmlc_calls_v4 c ON c.GMLC_JOIN_KEY=p.GMLC_JOIN_KEY
    WHERE p.RAW_JOIN_KEY<>''
    """
)
con.execute("CREATE INDEX ix_bridge_pair_key_v4 ON ims_bridge_pairs_v4(KEY_TYPE, RAW_JOIN_KEY)")
con.execute("CREATE INDEX ix_bridge_pair_call_v4 ON ims_bridge_pairs_v4(CALL_KEY)")
con.execute("DROP TABLE IF EXISTS ims_bridge_key_cardinality_v4")
con.execute(
    """
    CREATE TABLE ims_bridge_key_cardinality_v4 AS
    SELECT KEY_TYPE, RAW_JOIN_KEY, COUNT(DISTINCT CALL_KEY) AS CALL_COUNT,
           GROUP_CONCAT(DISTINCT CALL_KEY) AS CALL_KEYS
    FROM ims_bridge_pairs_v4
    GROUP BY KEY_TYPE, RAW_JOIN_KEY
    """
)
con.execute("DROP TABLE IF EXISTS ims_bridge_conflicts_v4")
con.execute(
    """CREATE TABLE ims_bridge_conflicts_v4 AS
       SELECT * FROM ims_bridge_key_cardinality_v4 WHERE CALL_COUNT>1"""
)
con.execute("DROP TABLE IF EXISTS ims_bridge_safe_v4")
con.execute(
    """
    CREATE TABLE ims_bridge_safe_v4 AS
    SELECT DISTINCT p.CALL_KEY, p.KEY_TYPE, p.RAW_JOIN_KEY
    FROM ims_bridge_pairs_v4 p
    JOIN ims_bridge_key_cardinality_v4 k
      ON k.KEY_TYPE=p.KEY_TYPE AND k.RAW_JOIN_KEY=p.RAW_JOIN_KEY
    WHERE k.CALL_COUNT=1
    """
)
con.execute("CREATE UNIQUE INDEX ux_safe_bridge_v4 ON ims_bridge_safe_v4(KEY_TYPE, RAW_JOIN_KEY, CALL_KEY)")
con.execute("CREATE INDEX ix_safe_bridge_call_v4 ON ims_bridge_safe_v4(CALL_KEY)")
con.commit()

ims_eligible_calls = con.execute("SELECT COUNT(*) FROM gmlc_calls_v4 WHERE GMLC_JOIN_KEY <> ''").fetchone()[0]
ims_matched_calls = con.execute(
    """SELECT COUNT(*) FROM gmlc_calls_v4 c
       WHERE c.GMLC_JOIN_KEY <> '' AND EXISTS
       (SELECT 1 FROM ims_agg_v4 i WHERE i.__KEY = c.GMLC_JOIN_KEY)"""
).fetchone()[0]
ims_duplicate_keys = con.execute("SELECT COUNT(*) FROM ims_agg_v4 WHERE IMS__ROW_COUNT > 1").fetchone()[0]
ims_coverage = ims_matched_calls / ims_eligible_calls if ims_eligible_calls else 0.0
if ims_coverage < MIN_JOIN_COVERAGE_WARNING:
    print(f"WARNING: exact GMLC→IMS coverage is only {ims_coverage:.1%}")


## 5. Exact RAW_CCDR enrichment

Both audited identifier pairs are evaluated when present:

- IMS `IMSCHARGINGID` = RAW `ICID`
- IMS `SESSIONID` = RAW `SESSION_ID`

Every IMS call↔identifier pair is preserved. Bridge identifiers that map to multiple calls are quarantined. RAW events are mapped through **both** safe identifiers, events resolving to different calls are quarantined, and all remaining RAW events are aggregated to one row per canonical call. Time proximity is never used as a final join.

In [ ]:
raw_key_columns = [
    ("ICID", "ICID"),
    ("SESSION_ID", "SESSION_ID"),
]
raw_key_columns = [pair for pair in raw_key_columns if pair[1] in headers["RAW_CCDR"]]
if not raw_key_columns:
    raise ValueError("RAW_CCDR has neither ICID nor SESSION_ID")

con.execute("DROP TABLE IF EXISTS raw_call_partial_v4")
create_categorical_partial_table(con, "raw_categorical_counts_partial_v4")
bridge_conflict_count = int(
    con.execute("SELECT COUNT(*) FROM ims_bridge_conflicts_v4").fetchone()[0]
)
bridge_conflict_path = OUTPUT_DIR / "bridge_conflicts_v4.csv"
if bridge_conflict_path.exists():
    bridge_conflict_path.unlink()
first_bridge_chunk = True
for bridge_chunk in pd.read_sql_query(
    "SELECT * FROM ims_bridge_conflicts_v4 ORDER BY CALL_COUNT DESC, KEY_TYPE, RAW_JOIN_KEY",
    con,
    chunksize=EXPORT_CHUNK_ROWS,
):
    bridge_chunk.to_csv(
        bridge_conflict_path,
        mode="w" if first_bridge_chunk else "a",
        header=first_bridge_chunk,
        index=False,
    )
    first_bridge_chunk = False
if first_bridge_chunk:
    pd.read_sql_query(
        "SELECT * FROM ims_bridge_conflicts_v4 LIMIT 0", con
    ).to_csv(bridge_conflict_path, index=False)

raw_conflict_path = OUTPUT_DIR / "raw_event_mapping_conflicts_v4.csv"
pd.DataFrame(columns=[
    "RAW_ROW_ID", "ICID", "SESSION_ID", "CANDIDATE_CALL_COUNT", "CANDIDATE_CALL_KEYS"
]).to_csv(raw_conflict_path, index=False)

raw_wanted = requested_by_source["RAW_CCDR"]
raw_rows_read = 0
raw_rows_with_safe_mapping = 0
raw_event_conflict_count = 0
raw_matches_by_key_type = Counter()
for chunk_number, chunk in enumerate(iter_selected_csv(
    paths["RAW_CCDR"], raw_wanted, SOURCE_CHUNK_ROWS
), start=1):
    chunk = chunk.reset_index(drop=True)
    chunk_start = raw_rows_read
    raw_rows_read += len(chunk)
    for column in RAW_ID_COLS:
        if column in chunk:
            chunk[column] = norm_id_series(chunk[column])

    raw_long_parts = []
    for key_type, raw_column in raw_key_columns:
        key = norm_id_series(chunk[raw_column])
        part = pd.DataFrame({
            "__RAW_POS": np.arange(len(chunk), dtype=np.int64),
            "KEY_TYPE": key_type,
            "RAW_JOIN_KEY": key,
        })
        raw_long_parts.append(part[part["RAW_JOIN_KEY"].ne("")])
    raw_long = pd.concat(raw_long_parts, ignore_index=True) if raw_long_parts else pd.DataFrame()
    if raw_long.empty:
        continue

    raw_long[["KEY_TYPE", "RAW_JOIN_KEY"]].drop_duplicates().to_sql(
        "raw_chunk_keys_v4", con, if_exists="replace", index=False
    )
    mapping = pd.read_sql_query(
        """
        SELECT k.KEY_TYPE, k.RAW_JOIN_KEY, b.CALL_KEY
        FROM raw_chunk_keys_v4 k
        JOIN ims_bridge_safe_v4 b
          ON b.KEY_TYPE=k.KEY_TYPE AND b.RAW_JOIN_KEY=k.RAW_JOIN_KEY
        """,
        con,
    )
    mapped_long = raw_long.merge(mapping, on=["KEY_TYPE", "RAW_JOIN_KEY"], how="inner")
    if mapped_long.empty:
        continue

    for key_type, count in mapped_long.groupby("KEY_TYPE")["__RAW_POS"].nunique().items():
        raw_matches_by_key_type[str(key_type)] += int(count)
    map_stats = mapped_long.groupby("__RAW_POS")["CALL_KEY"].agg(
        CANDIDATE_CALL_COUNT="nunique",
        CANDIDATE_CALL_KEYS=lambda s: "|".join(sorted(pd.unique(s.astype("string")))),
    )
    conflict_stats = map_stats[map_stats["CANDIDATE_CALL_COUNT"] > 1].reset_index()
    if not conflict_stats.empty:
        conflict_stats["RAW_ROW_ID"] = chunk_start + conflict_stats["__RAW_POS"] + 1
        conflict_stats["ICID"] = conflict_stats["__RAW_POS"].map(
            chunk["ICID"] if "ICID" in chunk else pd.Series("", index=chunk.index)
        )
        conflict_stats["SESSION_ID"] = conflict_stats["__RAW_POS"].map(
            chunk["SESSION_ID"] if "SESSION_ID" in chunk else pd.Series("", index=chunk.index)
        )
        conflict_stats[[
            "RAW_ROW_ID", "ICID", "SESSION_ID", "CANDIDATE_CALL_COUNT", "CANDIDATE_CALL_KEYS"
        ]].to_csv(raw_conflict_path, mode="a", header=False, index=False)
        raw_event_conflict_count += len(conflict_stats)

    safe_positions = map_stats.index[map_stats["CANDIDATE_CALL_COUNT"] == 1]
    safe_map = (
        mapped_long[mapped_long["__RAW_POS"].isin(safe_positions)]
        .sort_values(["__RAW_POS", "KEY_TYPE"])
        .drop_duplicates("__RAW_POS")[["__RAW_POS", "CALL_KEY"]]
        .reset_index(drop=True)
    )
    if not safe_map.empty:
        safe_chunk = chunk.iloc[safe_map["__RAW_POS"].to_numpy()].reset_index(drop=True)
        partial = aggregate_source_chunk(
            safe_chunk, safe_map["CALL_KEY"].reset_index(drop=True), "RAW",
            RAW_ID_COLS, RAW_CAT_COLS, RAW_NUM_COLS, RAW_PRESENCE_COLS,
            con, "raw_categorical_counts_partial_v4",
        )
        if not partial.empty:
            partial.to_sql("raw_call_partial_v4", con, if_exists="append", index=False, chunksize=5_000)
        raw_rows_with_safe_mapping += len(safe_map)
    if chunk_number == 1 or raw_rows_read % 500_000 < len(chunk):
        print(
            f"RAW_CCDR read {raw_rows_read:,}; safe event mappings {raw_rows_with_safe_mapping:,}; "
            f"conflicts {raw_event_conflict_count:,}"
        )

if "raw_call_partial_v4" not in {
    row[0] for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'")
}:
    raise ValueError("No RAW_CCDR events mapped through a safe exact IMS bridge")
finalize_partial_table(
    con,
    "raw_call_partial_v4",
    "raw_categorical_counts_partial_v4",
    "raw_call_agg_v4",
    "RAW",
)
raw_eligible_calls = con.execute(
    "SELECT COUNT(DISTINCT CALL_KEY) FROM ims_bridge_safe_v4"
).fetchone()[0]
raw_matched_calls = con.execute("SELECT COUNT(*) FROM raw_call_agg_v4").fetchone()[0]
raw_coverage = raw_matched_calls / raw_eligible_calls if raw_eligible_calls else 0.0
raw_duplicate_call_keys = con.execute(
    "SELECT COUNT(*) FROM (SELECT __KEY FROM raw_call_agg_v4 GROUP BY __KEY HAVING COUNT(*)>1)"
).fetchone()[0]
if raw_duplicate_call_keys:
    raise AssertionError("RAW call aggregate is not unique by canonical CALL_KEY")
if raw_coverage < MIN_JOIN_COVERAGE_WARNING:
    print(f"WARNING: combined exact IMS→RAW call coverage is only {raw_coverage:.1%}")
print("RAW exact bridges used together:", dict(raw_matches_by_key_type))


## 6. Build one-row-per-call enriched output and join audit

The final query joins only unique call-level aggregate tables. It asserts that the enriched row count exactly equals the canonical call count before writing the CSV.

In [ ]:
join_diagnostics = pd.DataFrame([
    {
        "JOIN_STAGE": "GMLC_TO_IMS",
        "JOIN_CANDIDATE": "UNIQ911_CID",
        "LEFT_COLUMN": "GMLC_JOIN_KEY",
        "RIGHT_COLUMN": "UNIQ911_CID",
        "ELIGIBLE_LEFT_CALLS": ims_eligible_calls,
        "MATCHED_LEFT_CALLS": ims_matched_calls,
        "MATCH_COVERAGE": ims_coverage,
        "RIGHT_KEYS_WITH_MULTIPLE_EVENTS": ims_duplicate_keys,
        "AGGREGATED_TO_ONE_ROW_PER_KEY": True,
        "SELECTED": True,
    },
    {
        "JOIN_STAGE": "IMS_TO_RAW",
        "JOIN_CANDIDATE": "ALL_SAFE_ICID_AND_SESSION_ID_PAIRS",
        "LEFT_COLUMN": "IMSCHARGINGID|SESSIONID",
        "RIGHT_COLUMN": "ICID|SESSION_ID",
        "ELIGIBLE_LEFT_CALLS": raw_eligible_calls,
        "MATCHED_LEFT_CALLS": raw_matched_calls,
        "MATCH_COVERAGE": raw_coverage,
        "RIGHT_KEYS_WITH_MULTIPLE_CALLS_QUARANTINED": bridge_conflict_count,
        "RAW_EVENTS_RESOLVING_TO_MULTIPLE_CALLS_QUARANTINED": raw_event_conflict_count,
        "AGGREGATED_TO_ONE_ROW_PER_KEY": True,
        "SELECTED": True,
    },
])
join_diagnostics.to_csv(OUTPUT_DIR / "join_diagnostics_v4.csv", index=False)
display(join_diagnostics)

ims_output_columns = [c for c in table_columns(con, "ims_agg_v4") if c != "__KEY"]
raw_output_columns = [c for c in table_columns(con, "raw_call_agg_v4") if c != "__KEY"]
ims_select = ", ".join(f"i.{quote_ident(c)} AS {quote_ident(c)}" for c in ims_output_columns)
raw_select = ", ".join(f"r.{quote_ident(c)} AS {quote_ident(c)}" for c in raw_output_columns)
extra_select = ", ".join(value for value in [ims_select, raw_select] if value)
if extra_select:
    extra_select = ", " + extra_select

FINAL_SQL = f"""
    SELECT c.* {extra_select}
    FROM gmlc_calls_v4 c
    LEFT JOIN ims_agg_v4 i ON i.__KEY = c.GMLC_JOIN_KEY
    LEFT JOIN raw_call_agg_v4 r ON r.__KEY = c.CALL_KEY
"""

final_count = con.execute(f"SELECT COUNT(*) FROM ({FINAL_SQL})").fetchone()[0]
if final_count != staged_call_count:
    raise AssertionError(
        f"Final joins multiplied/dropped canonical calls: input={staged_call_count:,}, final={final_count:,}"
    )

def export_query_csv(connection, query: str, path: Path, chunksize: int = EXPORT_CHUNK_ROWS) -> int:
    if path.exists():
        path.unlink()
    rows = 0
    first = True
    for frame in pd.read_sql_query(query, connection, chunksize=chunksize):
        frame.to_csv(path, mode="w" if first else "a", header=first, index=False)
        first = False
        rows += len(frame)
    if first:
        pd.read_sql_query(f"SELECT * FROM ({query}) LIMIT 0", connection).to_csv(path, index=False)
    return rows

labeled_export_rows = export_query_csv(
    con, "SELECT * FROM gmlc_calls_v4 ORDER BY GMLC_ROW_ID",
    OUTPUT_DIR / "gmlc_calls_with_boundary_labels_v4.csv",
)
enriched_export_rows = export_query_csv(
    con, FINAL_SQL + " ORDER BY c.GMLC_ROW_ID", OUTPUT_DIR / "enriched_calls_v4.csv",
)
definite_misroute_export_rows = export_query_csv(
    con,
    f"SELECT * FROM ({FINAL_SQL}) WHERE ROUTE_INTEGRITY_STATUS='DEFINITE_MISROUTE' ORDER BY GMLC_ROW_ID",
    OUTPUT_DIR / "definite_misroutes_v4.csv",
)
if labeled_export_rows != staged_call_count or enriched_export_rows != staged_call_count:
    raise AssertionError("CSV output reconciliation failed")
if definite_misroute_export_rows != con.execute(
    "SELECT COUNT(*) FROM gmlc_calls_v4 WHERE ROUTE_INTEGRITY_STATUS='DEFINITE_MISROUTE'"
).fetchone()[0]:
    raise AssertionError("Definite-misroute export reconciliation failed")
print(f"One-row-per-canonical-call invariant passed: {enriched_export_rows:,} calls")


## 7. Automated streaming root-cause analysis

RCA compares only strong labels (`DEFINITE_CORRECT` vs `DEFINITE_MISROUTE`) while retaining all statuses in the enriched call file. Categorical evidence is ranked by excess misroutes above the global baseline. Numeric counts, means, minima, and maxima are exact over non-null values.

In [ ]:
CAT_FEATURES = [
    "CALL_POPULATION_TAG", "REGION", "MARKET", "MARKET_CLUSTER", "STATE", "COUNTY",
    "GMLC_VENDOR", "MME_NAME", "MME_VENDOR", "MME_POOL_ID", "MME_POOL_NAME",
    "SECTOR_NAME", "TAC", "SETUP_ECGI_HEX", "ESRK", "USID", "UE_MAKE_ABBR",
    "UE_MODEL", "POS_METHOD_USED", "CALL_ANOMALY", "CALL_STRATEGY", "CALL_TRIGGER",
    "CALL_TYPE", "COMPLETE_CALL", "DEFAULT_ROUTED_CALL", "TAC_ROUTED_CALL",
    "ELS_AVAILABLE", "ELS_USED", "EXCEPTION_FLAG", "EXCEPTION_TYPE", "FAILURE_SHORT_DR",
    "LSR_EXCLUDE_REASON", "HDV_RESULT_CODE", "LRRINVITE_STATUS", "MLLSTATUS_CODE",
    "NW_METHOD_USED", "ORIG_RESULT_CODE", "P2_FAILURE_SHORT", "P2_LOCATE", "P2_SUCCESS",
    "PHASE", "PL_RESULT_CODE", "ROUTE_ESINET", "ROUTE_ESZ", "ROUTE_FALLBACK",
    "ROUTE_LOCATION_POS_SOURCE", "ROUTE_OUT_TYPE", "ROUTE_STATUS_GMLC", "SHAPE_TYPE",
    "SIGNALLING_TYPE", "SIPCELL_TYPE", "SIP_STATUS", "SIP_SIP_METHOD", "USER_TYPE",
    "IMS__RAN_REGION", "IMS__RAN_MARKET", "IMS__RAN_MARKET_CLUSTER", "IMS__CLASSIFICATION",
    "IMS__ESINET_PROVIDER", "IMS__MSC_CLLI", "IMS__PGW_CLLI", "IMS__NE_SITE_STRING",
    "IMS__LOCATIONXY_YN", "IMS__REGISTER_PCSCF_REASONCODES", "IMS__REGISTER_PCSCF_STATUS",
    "IMS__RULESET", "IMS__SERVICE", "IMS__SIGNATURE", "IMS__SIP_METHOD", "IMS__SOFTDROPCALL",
    "RAW__CORRELATION_STATUS", "RAW__DEFAULT_ROUTED_CALL", "RAW__ECSCF_STATUS",
    "RAW__END_CELLSITE", "RAW__NODE_ADDRESS", "RAW__NON_REGISTER_PCSCF_STATUS",
    "RAW__RECORD_TYPE", "RAW__REGISTER_PCSCF_STATUS", "RAW__START_CELLSITE",
]
NUM_FEATURES = [
    "UNCERT_METERS", "DISTANCE_TO_ROUTED_BOUNDARY_M", "CALL_DURATION_SEC", "CONFIDENCE",
    "CALL_BEGIN_TO_LOCATE_SEC", "INVITE_TO_END_SEC", "LOCATE_TIME_SEC", "TIME_ON_LTE_SEC",
    "LBR_ATTEMPTED", "LBR_PSAP_DIFF", "LBR_SUCCESS", "INVITE_CNT", "LOCATE_CNT",
    "IMS__DURATION__MEAN", "IMS__CDRS_LEGS__MEAN",
    "IMS__COUNT_OF_REGISTER_PCSCF__MEAN",
    "RAW__COUNT_OF_ECSCF__MEAN",
    "RAW__COUNT_OF_NON_REGISTER_PCSCF__MEAN",
    "RAW__COUNT_OF_REGISTER_PCSCF__MEAN",
]
final_columns = set(table_columns(con, "gmlc_calls_v4")) | set(ims_output_columns) | set(raw_output_columns)
CAT_FEATURES = [c for c in dict.fromkeys(CAT_FEATURES) if c in final_columns]
NUM_FEATURES = [c for c in dict.fromkeys(NUM_FEATURES) if c in final_columns]
MISSINGNESS_FEATURES = list(dict.fromkeys(CAT_FEATURES + NUM_FEATURES + [
    c for c in final_columns if c.endswith("__PRESENT")
]))

# Keep high-cardinality RCA counts on disk. GMLC identifiers such as
# ECGI/USID/ESRK can have millions of distinct values, so an unbounded
# Python dictionary is not safe for the full-month exports.
con.execute("DROP TABLE IF EXISTS rca_categorical_partial_v4")
con.execute(
    """CREATE TABLE rca_categorical_partial_v4 (
           FEATURE TEXT NOT NULL,
           FEATURE_VALUE TEXT NOT NULL,
           TOTAL_STRONG_CALLS INTEGER NOT NULL,
           MISROUTE_CALLS INTEGER NOT NULL
       )"""
)
con.execute("DROP TABLE IF EXISTS rca_missingness_partial_v4")
con.execute(
    """CREATE TABLE rca_missingness_partial_v4 (
           FEATURE TEXT NOT NULL,
           AVAILABILITY TEXT NOT NULL,
           TOTAL_STRONG_CALLS INTEGER NOT NULL,
           MISROUTE_CALLS INTEGER NOT NULL
       )"""
)
numeric_stats = defaultdict(lambda: {
    0: {"count": 0, "sum": 0.0, "min": np.inf, "max": -np.inf},
    1: {"count": 0, "sum": 0.0, "min": np.inf, "max": -np.inf},
})
strong_total = 0
misroute_total = 0

rca_columns = list(dict.fromkeys(
    ["MISROUTE_LABEL"] + CAT_FEATURES + NUM_FEATURES + MISSINGNESS_FEATURES
))
gmlc_rca_columns = set(table_columns(con, "gmlc_calls_v4"))
ims_rca_columns = set(ims_output_columns)
raw_rca_columns = set(raw_output_columns)

def rca_column_expression(column: str) -> str:
    if column in gmlc_rca_columns:
        return f"c.{quote_ident(column)}"
    if column in ims_rca_columns:
        return f"i.{quote_ident(column)}"
    if column in raw_rca_columns:
        return f"r.{quote_ident(column)}"
    raise KeyError(f"RCA column has no source relation: {column}")

rca_projection = ", ".join(
    f"{rca_column_expression(c)} AS {quote_ident(c)}" for c in rca_columns
)
RCA_SQL = f"""
    SELECT {rca_projection}
    FROM gmlc_calls_v4 c
    LEFT JOIN ims_agg_v4 i ON i.__KEY=c.GMLC_JOIN_KEY
    LEFT JOIN raw_call_agg_v4 r ON r.__KEY=c.CALL_KEY
"""

for frame_number, frame in enumerate(
    pd.read_sql_query(RCA_SQL, con, chunksize=RCA_CHUNK_ROWS), start=1
):
    labels = pd.to_numeric(frame["MISROUTE_LABEL"], errors="coerce")
    strong_mask = labels.notna()
    strong = frame.loc[strong_mask].copy()
    y = labels.loc[strong_mask].astype(int)
    strong_total += len(strong)
    misroute_total += int(y.sum())

    for feature in CAT_FEATURES:
        values = strong[feature].astype("string").fillna("<MISSING>").str.strip().replace("", "<MISSING>")
        grouped = pd.DataFrame({"VALUE": values, "Y": y.values}).groupby("VALUE", dropna=False)["Y"].agg(["size", "sum"])
        if not grouped.empty:
            partial = grouped.reset_index().rename(columns={
                "VALUE": "FEATURE_VALUE", "size": "TOTAL_STRONG_CALLS",
                "sum": "MISROUTE_CALLS",
            })
            partial.insert(0, "FEATURE", feature)
            partial.to_sql(
                "rca_categorical_partial_v4", con, if_exists="append",
                index=False, chunksize=5_000,
            )

    for feature in MISSINGNESS_FEATURES:
        values = strong[feature]
        if feature.endswith("__PRESENT"):
            present = pd.to_numeric(values, errors="coerce").fillna(0).eq(1)
            missing = ~present
        else:
            missing = values.isna() | values.astype("string").fillna("").str.strip().eq("")
        rows = []
        for state, mask in (("MISSING", missing), ("PRESENT", ~missing)):
            rows.append({
                "FEATURE": feature,
                "AVAILABILITY": state,
                "TOTAL_STRONG_CALLS": int(mask.sum()),
                "MISROUTE_CALLS": int(y[mask].sum()) if mask.any() else 0,
            })
        pd.DataFrame(rows).to_sql(
            "rca_missingness_partial_v4", con, if_exists="append",
            index=False,
        )

    for feature in NUM_FEATURES:
        values = pd.to_numeric(strong[feature], errors="coerce")
        for cls in (0, 1):
            data = values[y.eq(cls)].dropna().to_numpy(dtype=float)
            if not len(data):
                continue
            stats = numeric_stats[feature][cls]
            stats["count"] += len(data)
            stats["sum"] += float(data.sum())
            stats["min"] = min(stats["min"], float(data.min()))
            stats["max"] = max(stats["max"], float(data.max()))

    if frame_number == 1 or frame_number % 10 == 0:
        print(f"RCA streamed {frame_number * RCA_CHUNK_ROWS:,} maximum rows")

baseline = misroute_total / strong_total if strong_total else np.nan
con.execute(
    "CREATE INDEX IF NOT EXISTS ix_rca_cat_partial_v4 "
    "ON rca_categorical_partial_v4(FEATURE, FEATURE_VALUE)"
)
con.execute(
    "CREATE INDEX IF NOT EXISTS ix_rca_missing_partial_v4 "
    "ON rca_missingness_partial_v4(FEATURE, AVAILABILITY)"
)
con.execute("DROP TABLE IF EXISTS categorical_lift_v4")
con.execute(
    f"""
    CREATE TABLE categorical_lift_v4 AS
    SELECT FEATURE, FEATURE_VALUE,
           SUM(TOTAL_STRONG_CALLS) AS TOTAL_STRONG_CALLS,
           SUM(MISROUTE_CALLS) AS MISROUTE_CALLS,
           100.0 * SUM(MISROUTE_CALLS) / SUM(TOTAL_STRONG_CALLS) AS MISROUTE_RATE_PCT,
           CASE WHEN {float(baseline) if np.isfinite(baseline) else 0.0} > 0
                THEN (1.0 * SUM(MISROUTE_CALLS) / SUM(TOTAL_STRONG_CALLS)) /
                     {float(baseline) if np.isfinite(baseline) else 1.0}
           END AS RATE_LIFT_VS_GLOBAL,
           SUM(TOTAL_STRONG_CALLS) * {float(baseline) if np.isfinite(baseline) else 0.0}
               AS EXPECTED_MISROUTES_AT_BASELINE,
           SUM(MISROUTE_CALLS) -
               SUM(TOTAL_STRONG_CALLS) * {float(baseline) if np.isfinite(baseline) else 0.0}
               AS EXCESS_MISROUTES
    FROM rca_categorical_partial_v4
    GROUP BY FEATURE, FEATURE_VALUE
    HAVING SUM(TOTAL_STRONG_CALLS) >= {RCA_MIN_STRONG_CALLS}
       AND SUM(MISROUTE_CALLS) >= {RCA_MIN_MISROUTES}
    """
)
entity_tokens = (
    "ECGI", "USID", "ESRK", "REGION", "MARKET", "COUNTY", "STATE",
    "CLLI", "SITE", "VENDOR", "MME", "SECTOR", "TAC",
)
signaling_tokens = (
    "STATUS", "RESULT", "FAIL", "EXCEPTION", "METHOD", "ROUTE",
    "LBR", "P2_", "SIGNAL", "ELS_",
)

def feature_token_predicate(tokens: Sequence[str]) -> str:
    return " OR ".join(
        "INSTR(UPPER(FEATURE), '" + token.replace("'", "''").upper() + "') > 0"
        for token in tokens
    ) or "0"

con.execute("DROP TABLE IF EXISTS entity_rca_v4")
con.execute(
    "CREATE TABLE entity_rca_v4 AS SELECT * FROM categorical_lift_v4 WHERE "
    + feature_token_predicate(entity_tokens)
)
con.execute("DROP TABLE IF EXISTS signaling_status_rca_v4")
con.execute(
    "CREATE TABLE signaling_status_rca_v4 AS SELECT * FROM categorical_lift_v4 WHERE "
    + feature_token_predicate(signaling_tokens)
)
con.execute("DROP TABLE IF EXISTS missingness_lift_v4")
con.execute(
    """
    CREATE TABLE missingness_lift_v4 AS
    SELECT FEATURE, AVAILABILITY,
           SUM(TOTAL_STRONG_CALLS) AS TOTAL_STRONG_CALLS,
           SUM(MISROUTE_CALLS) AS MISROUTE_CALLS,
           100.0 * SUM(MISROUTE_CALLS) / NULLIF(SUM(TOTAL_STRONG_CALLS), 0)
               AS MISROUTE_RATE_PCT
    FROM rca_missingness_partial_v4
    GROUP BY FEATURE, AVAILABILITY
    """
)
con.commit()
categorical_lift = pd.read_sql_query(
    """SELECT * FROM categorical_lift_v4
       ORDER BY EXCESS_MISROUTES DESC, RATE_LIFT_VS_GLOBAL DESC
       LIMIT 1000""",
    con,
)
missingness_lift = pd.read_sql_query(
    "SELECT * FROM missingness_lift_v4 ORDER BY FEATURE, AVAILABILITY",
    con,
)
export_query_csv(
    con,
    """SELECT * FROM categorical_lift_v4
       ORDER BY EXCESS_MISROUTES DESC, RATE_LIFT_VS_GLOBAL DESC""",
    OUTPUT_DIR / "categorical_lift_v4.csv",
)
export_query_csv(
    con,
    "SELECT * FROM missingness_lift_v4 ORDER BY FEATURE, AVAILABILITY",
    OUTPUT_DIR / "missingness_lift_v4.csv",
)
export_query_csv(
    con,
    """SELECT * FROM entity_rca_v4
       ORDER BY EXCESS_MISROUTES DESC, RATE_LIFT_VS_GLOBAL DESC""",
    OUTPUT_DIR / "entity_rca_v4.csv",
)
export_query_csv(
    con,
    """SELECT * FROM signaling_status_rca_v4
       ORDER BY EXCESS_MISROUTES DESC, RATE_LIFT_VS_GLOBAL DESC""",
    OUTPUT_DIR / "signaling_status_rca_v4.csv",
)
entity_rca = pd.read_sql_query(
    """SELECT * FROM entity_rca_v4
       ORDER BY EXCESS_MISROUTES DESC, RATE_LIFT_VS_GLOBAL DESC LIMIT 75""",
    con,
)
signaling_rca = pd.read_sql_query(
    """SELECT * FROM signaling_status_rca_v4
       ORDER BY EXCESS_MISROUTES DESC, RATE_LIFT_VS_GLOBAL DESC LIMIT 75""",
    con,
)

numeric_rows = []
for feature, by_class in numeric_stats.items():
    correct, misroute = by_class[0], by_class[1]
    if correct["count"] < 20 or misroute["count"] < 5:
        continue
    numeric_rows.append({
        "FEATURE": feature,
        "CORRECT_NON_NULL": correct["count"], "MISROUTE_NON_NULL": misroute["count"],
        "CORRECT_MEAN": correct["sum"] / correct["count"],
        "MISROUTE_MEAN": misroute["sum"] / misroute["count"],
        "MEAN_DIFFERENCE": (misroute["sum"] / misroute["count"]) - (correct["sum"] / correct["count"]),
        "CORRECT_MIN": correct["min"], "MISROUTE_MIN": misroute["min"],
        "CORRECT_MAX": correct["max"], "MISROUTE_MAX": misroute["max"],
    })
numeric_comparison = pd.DataFrame(numeric_rows, columns=[
    "FEATURE", "CORRECT_NON_NULL", "MISROUTE_NON_NULL", "CORRECT_MEAN",
    "MISROUTE_MEAN", "MEAN_DIFFERENCE", "CORRECT_MIN", "MISROUTE_MIN",
    "CORRECT_MAX", "MISROUTE_MAX",
])


In [ ]:
population_summary = pd.read_sql_query(
    """SELECT CALL_POPULATION_TAG,
              COUNT(*) AS TOTAL_CALLS,
              SUM(CASE WHEN MISROUTE_LABEL IS NOT NULL THEN 1 ELSE 0 END) AS STRONG_CALLS,
              SUM(CASE WHEN MISROUTE_LABEL = 1 THEN 1 ELSE 0 END) AS MISROUTE_CALLS,
              ROUND(100.0 * SUM(CASE WHEN MISROUTE_LABEL = 1 THEN 1 ELSE 0 END) /
                    NULLIF(SUM(CASE WHEN MISROUTE_LABEL IS NOT NULL THEN 1 ELSE 0 END), 0), 6)
                    AS MISROUTE_RATE_PCT
       FROM gmlc_calls_v4 GROUP BY CALL_POPULATION_TAG ORDER BY TOTAL_CALLS DESC""",
    con,
)
route_pair_rca = pd.read_sql_query(
    """SELECT FCC_PSAP_ID, EXPECTED_FCC_PSAP_ID, CALL_POPULATION_TAG,
              COUNT(*) AS TOTAL_STRONG_CALLS,
              SUM(CASE WHEN MISROUTE_LABEL = 1 THEN 1 ELSE 0 END) AS MISROUTE_CALLS,
              MIN(CALL_BEGIN_TIME_UTC) AS FIRST_CALL_UTC,
              MAX(CALL_BEGIN_TIME_UTC) AS LAST_CALL_UTC,
              ROUND(100.0 * SUM(CASE WHEN MISROUTE_LABEL = 1 THEN 1 ELSE 0 END) / COUNT(*), 6)
                  AS MISROUTE_RATE_PCT
       FROM gmlc_calls_v4
       WHERE MISROUTE_LABEL IS NOT NULL
       GROUP BY FCC_PSAP_ID, EXPECTED_FCC_PSAP_ID, CALL_POPULATION_TAG
       ORDER BY MISROUTE_CALLS DESC, TOTAL_STRONG_CALLS DESC""",
    con,
)

episode_group_cols = [c for c in [
    "FCC_PSAP_ID", "EXPECTED_FCC_PSAP_ID", "SETUP_ECGI_HEX", "USID", "ESRK", "CALL_POPULATION_TAG"
] if c in table_columns(con, "gmlc_calls_v4")]
episode_select = ", ".join(quote_ident(c) for c in episode_group_cols)
episode_rca = pd.read_sql_query(
    f"""SELECT {episode_select}, COUNT(*) AS TOTAL_STRONG_CALLS,
               SUM(CASE WHEN MISROUTE_LABEL = 1 THEN 1 ELSE 0 END) AS MISROUTE_CALLS,
               MIN(CALL_BEGIN_TIME_UTC) AS FIRST_CALL_UTC,
               MAX(CALL_BEGIN_TIME_UTC) AS LAST_CALL_UTC,
               ROUND(100.0 * SUM(CASE WHEN MISROUTE_LABEL = 1 THEN 1 ELSE 0 END) / COUNT(*), 6)
                   AS MISROUTE_RATE_PCT
        FROM gmlc_calls_v4 WHERE MISROUTE_LABEL IS NOT NULL
        GROUP BY {episode_select}
        HAVING SUM(CASE WHEN MISROUTE_LABEL = 1 THEN 1 ELSE 0 END) >= {RCA_MIN_MISROUTES}
        ORDER BY MISROUTE_CALLS DESC, TOTAL_STRONG_CALLS DESC""",
    con,
)

# Entity/signaling CSVs were already exported from the complete SQLite
# tables. Only the bounded previews remain in pandas for display.
evidence_outputs = {
    "population_summary_v4.csv": population_summary,
    "route_pair_rca_v4.csv": route_pair_rca,
    "episode_rca_v4.csv": episode_rca,
    "numeric_comparison_v4.csv": numeric_comparison,
}
for filename, frame in evidence_outputs.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)


## 8. Results

Start with route pairs and repeated episodes, then use entity and signaling tables to explain where those misroutes concentrate. Test/sim calls remain visible through `CALL_POPULATION_TAG`.

In [ ]:
display(Markdown("### Label distribution"))
display(label_summary)
display(Markdown("### Population tags — no calls excluded"))
display(population_summary)
display(Markdown("### Highest-volume routed → expected PSAP patterns"))
display(route_pair_rca.head(50))
display(Markdown("### Repeated misroute episodes"))
display(episode_rca.head(50))
display(Markdown("### Network/entity root-cause candidates"))
display(entity_rca.head(75))
display(Markdown("### Signaling, routing, and failure associations"))
display(signaling_rca.head(75))
display(Markdown("### Numeric comparison"))
display(numeric_comparison)
display(Markdown("### Highest observed categorical excess"))
display(categorical_lift.head(75))


## 9. Output reconciliation and end marker

The manifest records the four source files, source-row and canonical-call invariants, label totals, exact-join quarantine counts, QA gates, and the complete output inventory. The notebook ends only after every gate passes.

In [ ]:
output_names = [
    "schema_validation_v4.csv",
    "boundary_parse_status_v4.csv",
    "call_grain_audit_v4.csv",
    "gmlc_calls_with_boundary_labels_v4.csv",
    "enriched_calls_v4.csv",
    "definite_misroutes_v4.csv",
    "join_diagnostics_v4.csv",
    "bridge_conflicts_v4.csv",
    "raw_event_mapping_conflicts_v4.csv",
    "population_summary_v4.csv",
    "route_pair_rca_v4.csv",
    "episode_rca_v4.csv",
    "entity_rca_v4.csv",
    "signaling_status_rca_v4.csv",
    "categorical_lift_v4.csv",
    "numeric_comparison_v4.csv",
    "missingness_lift_v4.csv",
    "run_manifest_v4.json",
]

definite_correct = int(con.execute("SELECT COUNT(*) FROM gmlc_calls_v4 WHERE MISROUTE_LABEL = 0").fetchone()[0])
definite_misroute = int(con.execute("SELECT COUNT(*) FROM gmlc_calls_v4 WHERE MISROUTE_LABEL = 1").fetchone()[0])
tagged_calls = int(con.execute("SELECT COUNT(*) FROM gmlc_calls_v4 WHERE IS_TEST_OR_SIM = 1").fetchone()[0])
conflicting_calls = int(con.execute("SELECT COUNT(*) FROM gmlc_calls_v4 WHERE CALL_CONFLICT=1").fetchone()[0])

qa_gates = {
    "gmlc_source_rows_staged_without_loss": bool(input_gmlc_rows == staged_gmlc_rows == staged_check),
    "canonical_call_count_reconciles": bool(staged_call_count == expected_call_count),
    "canonical_call_key_unique": bool(duplicate_call_keys == 0),
    "conflicting_calls_have_no_strong_label": bool(conflict_strong_labels == 0),
    "raw_call_aggregate_unique": bool(raw_duplicate_call_keys == 0),
    "final_join_preserves_canonical_calls": bool(final_count == staged_call_count),
    "labeled_export_reconciles": bool(labeled_export_rows == staged_call_count),
    "enriched_export_reconciles": bool(enriched_export_rows == staged_call_count),
    "definite_misroute_export_reconciles": bool(definite_misroute_export_rows == definite_misroute),
}
if not all(qa_gates.values()):
    raise AssertionError(f"One or more v4 QA gates failed: {qa_gates}")

manifest = {
    "pipeline": "psap_gmlc_misroute_rca_v4",
    "call_grain": "one call per normalized UNIQ911_CID; missing CID uses ROW:<GMLC_ROW_ID>",
    "sources": {
        source: {
            "path": str(path),
            "size_bytes": int(path.stat().st_size),
            "columns": int(len(headers[source])),
        }
        for source, path in paths.items()
    },
    "input_gmlc_source_rows": int(input_gmlc_rows),
    "staged_gmlc_source_rows": int(staged_gmlc_rows),
    "canonical_calls": int(staged_call_count),
    "final_enriched_calls": int(final_count),
    "conflicting_duplicate_cid_calls_quarantined": conflicting_calls,
    "usable_boundaries": int(len(boundaries)),
    "definite_correct": definite_correct,
    "definite_misroute": definite_misroute,
    "strong_label_coverage_pct": 100 * (definite_correct + definite_misroute) / max(final_count, 1),
    "test_or_sim_calls_retained": tagged_calls,
    "gmlc_to_ims_match_coverage": float(ims_coverage),
    "raw_join_method": "all safe exact IMSCHARGINGID=ICID and SESSIONID=SESSION_ID pairs",
    "raw_match_coverage": float(raw_coverage),
    "bridge_keys_mapping_to_multiple_calls_quarantined": bridge_conflict_count,
    "raw_events_mapping_to_multiple_calls_quarantined": int(raw_event_conflict_count),
    "qa_gates": qa_gates,
    "output_directory": str(OUTPUT_DIR),
    "outputs": output_names,
}
manifest_path = OUTPUT_DIR / "run_manifest_v4.json"
with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

output_inventory = pd.DataFrame([
    {
        "FILE": name,
        "EXISTS": (OUTPUT_DIR / name).exists(),
        "SIZE_MB": round((OUTPUT_DIR / name).stat().st_size / 1024**2, 3)
        if (OUTPUT_DIR / name).exists() else None,
        "PATH": str(OUTPUT_DIR / name),
    }
    for name in output_names
])
display(output_inventory)
if not output_inventory["EXISTS"].all():
    raise AssertionError("One or more required v4 outputs are missing")
manifest["output_inventory"] = output_inventory.to_dict(orient="records")
with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

con.close()
display(Markdown(
    "# V4 NOTEBOOK COMPLETED SUCCESSFULLY\n\n"
    f"- Raw GMLC source rows staged: **{staged_gmlc_rows:,}**\n"
    f"- Canonical calls analyzed: **{final_count:,}**\n"
    f"- Definite correct: **{definite_correct:,}**\n"
    f"- Definite misroutes: **{definite_misroute:,}**\n"
    f"- Conflicting duplicate-CID calls quarantined: **{conflicting_calls:,}**\n"
    f"- Test/sim-tagged calls retained: **{tagged_calls:,}**\n"
    f"- Exact RAW mapping: **both safe bridge identifier families**\n"
    f"- Outputs: `{OUTPUT_DIR}`"
))
